<a href="https://colab.research.google.com/github/servantjoseph/Entropy_Balancing/blob/main/ACS_EBW_Boosted_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ACS EBW Boosted (Notebook)

This notebook is a direct conversion of `ACS_EBW_Boosted_code.py` into a runnable Jupyter notebook. It prepares the ACS data, constructs features, and computes biased sampling probabilities used for experiments with entropy balancing and hybrid tree-boosted entropy balancing.

In [1]:
import os
import numpy as np
import pandas as pd

# Download entropy_common.py if not already present
if not os.path.exists('entropy_common.py'):
    !wget https://raw.githubusercontent.com/servantjoseph/Entropy_Balancing/main/entropy_common.py


from entropy_common import (
    pairwise_products,
)

BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
DATA_PATH = os.path.join(BASE_DIR, "acs12.csv")
OUT_DIR = BASE_DIR
SEED = 20260625

# -----------------------------
# ACS Feature Engineering
# -----------------------------

def stable_softmax(z):
    z = np.asarray(z, dtype=float)
    z = z - np.max(z)
    p = np.exp(z)
    return p / p.sum()

def standardize_train_target(source_raw, target_raw):
    """Standardize source and target using target mean/sd."""
    mu = target_raw.mean(axis=0)
    sd = target_raw.std(axis=0)
    sd[sd < 1e-10] = 1.0
    return (source_raw - mu) / sd, (target_raw - mu) / sd, mu, sd

def drop_zero_variance(A, tol=1e-12):
    sd = A.std(axis=0)
    keep = sd > tol
    return A[:, keep], keep

def make_features(df):
    """Create raw feature matrices for main effects, compact pairwise basis, and tree search.
    """
    d = df.copy()
    # Numeric preprocessing for covariates only. Outcome is not used for weighting.
    d["hrs_work_imp"] = d["hrs_work"].fillna(0.0)
    d["hrs_work_missing"] = d["hrs_work"].isna().astype(float)
    d["time_to_work_imp"] = d["time_to_work"].fillna(0.0)
    d["time_to_work_missing"] = d["time_to_work"].isna().astype(float)
    d["lang_missing"] = d["lang"].isna().astype(float)
    d["edu_missing"] = d["edu"].isna().astype(float)
    d["lang"] = d["lang"].fillna("missing")
    d["edu"] = d["edu"].fillna("missing")

    # Include all covariate main effects. Drop first category for each factor.
    cat_cols = ["employment", "race", "gender", "citizen", "lang", "married", "edu", "disability", "birth_qrtr"]
    num_cols = ["age", "hrs_work_imp", "hrs_work_missing", "time_to_work_imp", "time_to_work_missing", "lang_missing", "edu_missing"]
    X_cat = pd.get_dummies(d[cat_cols], drop_first=True, dtype=float)
    X_num = d[num_cols].astype(float)
    X_main_df = pd.concat([X_num, X_cat], axis=1)

    # Compact features for pairwise products and tree search. Keep interpretable signals.
    compact = pd.DataFrame({
        "age": d["age"].astype(float),
        "hrs_work": d["hrs_work_imp"].astype(float),
        "commute": d["time_to_work_imp"].astype(float),
        "employed": (d["employment"] == "employed").astype(float),
        "male": (d["gender"] == "male").astype(float),
        "college": (d["edu"].isin(["college", "grad"]) ).astype(float),
        "grad": (d["edu"] == "grad").astype(float),
        "nonwhite": (d["race"] != "white").astype(float),
        "citizen": (d["citizen"] == "yes").astype(float),
        "english": (d["lang"] == "english").astype(float),
        "married": (d["married"] == "yes").astype(float),
        "disabled": (d["disability"] == "yes").astype(float),
    })
    return X_main_df, compact

# -----------------------------
# ACS Data Preparation
# -----------------------------

def ensure_acs_data():
    """Download the OpenIntro acs12 CSV if it is not already present.
    """
    if os.path.exists(DATA_PATH):
        return
    import urllib.request
    url = "https://www.openintro.org/data/csv/acs12.csv"
    urllib.request.urlretrieve(url, DATA_PATH)

def prepare_acs_inputs():
    """
    Prepare ACS data for analysis.

    Returns:
        dict: A dictionary containing:
            - N: population size
    ... (see script docstring for full list)
    """
    ensure_acs_data()
    df = pd.read_csv(DATA_PATH)
    df = df[(df["age"] >= 18) & df["income"].notna()].copy().reset_index(drop=True)
    df["log_income"] = np.log1p(df["income"].astype(float))
    N = len(df)

    X_main_df, X_compact_df = make_features(df)
    X_main_raw = X_main_df.to_numpy(dtype=float)
    X_comp_raw = X_compact_df.to_numpy(dtype=float)

    X_main_std, _, _, _ = standardize_train_target(X_main_raw, X_main_raw)
    X_main_std, keep_main = drop_zero_variance(X_main_std)
    main_names = [c for c, keep in zip(X_main_df.columns, keep_main) if keep]
    X_comp_std, _, _, _ = standardize_train_target(X_comp_raw, X_comp_raw)
    X_comp_std, keep_comp = drop_zero_variance(X_comp_std)
    comp_names = [c for c, keep in zip(X_compact_df.columns, keep_comp) if keep]

    mu_main = X_main_std.mean(axis=0)
    pair_prod = pairwise_products(X_comp_std[:, :8])
    pair_all = np.hstack([X_main_std, pair_prod])
    pair_all, keep_pair = drop_zero_variance(pair_all)
    mu_pair = pair_all.mean(axis=0)
    wt = np.ones(N) / N
    y = df["log_income"].to_numpy(dtype=float)
    y_income = df["income"].to_numpy(dtype=float)
    target_log = float(np.mean(y))
    target_income = float(np.mean(y_income))

    c = X_compact_df
    age_scaled = (c["age"].values - c["age"].mean()) / c["age"].std()
    hours_scaled = (c["hrs_work"].values - c["hrs_work"].mean()) / (c["hrs_work"].std() + 1e-8)
    commute_scaled = (c["commute"].values - c["commute"].mean()) / (c["commute"].std() + 1e-8)
    employed = c["employed"].values
    male = c["male"].values
    college = c["college"].values
    nonwhite = c["nonwhite"].values
    english = c["english"].values
    married = c["married"].values
    disabled = c["disabled"].values
    score = (
        0.12 * employed +
        0.10 * college +
        0.06 * male +
        0.06 * married -
        0.05 * disabled +
        0.04 * age_scaled +
        0.05 * hours_scaled -
        0.04 * commute_scaled +
        2.20 * (age_scaled > 0.70) * college +
        1.70 * (hours_scaled > 0.60) * employed +
        1.20 * (commute_scaled < -0.50) * male +
        1.00 * (age_scaled < -0.75) * (1 - english) -
        1.20 * nonwhite * (1 - english) +
        0.75 * employed * college * male
    )
    probs = stable_softmax(score)
    return {
        "N": N,
        "X_main_std": X_main_std,
        "X_comp_std": X_comp_std,
        "pair_all": pair_all,
        "mu_main": mu_main,
        "mu_pair": mu_pair,
        "wt": wt,
        "y": y,
        "y_income": y_income,
        "target_log": target_log,
        "target_income": target_income,
        "probs": probs,
        "main_names": main_names,
        "comp_names": comp_names,
    }

# Run example to prepare and display ACS data (if desired)
if __name__ == "__main__":
    acs_data = prepare_acs_inputs()
    print(f"Population size: {acs_data['N']}")
    print(f"Main effect features: {len(acs_data['main_names'])}")
    print(f"Compact features: {len(acs_data['comp_names'])}")
    print(f"Target mean log income: {acs_data['target_log']:.4f}")
    print(f"Target mean income: {acs_data['target_income']:.2f}")


--2026-08-25 19:21:35--  https://raw.githubusercontent.com/servantjoseph/Entropy_Balancing/main/entropy_common.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4073 (4.0K) [text/plain]
Saving to: ‘entropy_common.py’

entropy_common.py   100%[===================>]   3.98K  --.-KB/s    in 0s      

2026-08-25 19:21:35 (41.7 MB/s) - ‘entropy_common.py’ saved [4073/4073]



ImportError: cannot import name 'stable_softmax' from 'entropy_common' (/content/entropy_common.py)